In [5]:
import os, io, math
import pandas as pd
from tqdm import tqdm
from sqlalchemy import create_engine, text

USER      = "admin"
PASSWORD  = "admin"
HOST      = "127.0.0.1"
PORT      = "5432"
DB_NAME   = "dbt_db"

# CSVs live in faker/data/ — run this notebook from the faker/ directory
DATA_DIR  = "data"

In [6]:
def copy_from_dataframe(conn, df, table, schema="raw", chunk_size=500_000):
    raw_conn = conn.connection
    cursor   = raw_conn.cursor()

    cursor.execute(f'DROP TABLE IF EXISTS {schema}."{table}"')
    raw_conn.commit()

    df.head(0).to_sql(table, con=conn, schema=schema, index=False, if_exists="replace")
    raw_conn.commit()

    total_chunks = math.ceil(len(df) / chunk_size)

    with tqdm(total=len(df), unit="rows", desc=f"Loading {table}") as pbar:
        for i in range(total_chunks):
            chunk = df.iloc[i * chunk_size : (i + 1) * chunk_size]
            buf   = io.StringIO()
            chunk.to_csv(buf, index=False, header=False)
            buf.seek(0)
            cursor.copy_expert(
                f'COPY {schema}."{table}" FROM STDIN WITH (FORMAT CSV)',
                buf,
            )
            raw_conn.commit()
            pbar.update(len(chunk))

    cursor.close()
    print(f"Done — {len(df):,} rows loaded into {schema}.{table}")

In [7]:
TABLES = [
    "raw_chart_of_accounts",
    "raw_customers",
    "raw_products",
    "raw_employees",
    "raw_vendors",
    "raw_subscriptions",
    "raw_invoices",
    "raw_invoice_lines",
    "raw_payments",
    "raw_vendor_bills",
    "raw_payroll",
    "raw_journal_entries",
]

url    = f"postgresql+psycopg2://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"
engine = create_engine(url)

with engine.connect() as conn:
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw"))
    conn.commit()
    print("Schema 'raw' ready.\n")

    for table in TABLES:
        df = pd.read_csv(os.path.join(DATA_DIR, f"{table}.csv"))
        copy_from_dataframe(conn, df, table, schema="raw")
        print()

Schema 'raw' ready.



Loading raw_chart_of_accounts: 100%|██████████| 28/28 [00:00<00:00, 5119.69rows/s]


Done — 28 rows loaded into raw.raw_chart_of_accounts



Loading raw_customers: 100%|██████████| 500/500 [00:00<00:00, 38683.56rows/s]


Done — 500 rows loaded into raw.raw_customers



...

Done — 720 rows loaded into raw.raw_vendor_bills



Loading raw_payroll: 100%|██████████| 2860/2860 [00:00<00:00, 76578.46rows/s]


Done — 2,860 rows loaded into raw.raw_payroll



Loading raw_journal_entries: 100%|██████████| 44414/44414 [00:00<00:00, 98254.14rows/s]

Done — 44,414 rows loaded into raw.raw_journal_entries



In [4]:
print(f"{'Table':<35} {'Rows':>8}")
print("-" * 45)

with engine.connect() as conn:
    for table in TABLES:
        n = conn.execute(text(f'SELECT COUNT(*) FROM raw."{table}"')).scalar()
        print(f"  {table:<33} {n:>8,}")

Table                                   Rows
---------------------------------------------
  raw_chart_of_accounts                   27
  raw_customers                          500
  raw_products                             8
  raw_employees                           55
  raw_vendors                             15
  raw_subscriptions                      500
  raw_invoices                         9,862
  raw_invoice_lines                    9,862
  raw_payments                         8,413
  raw_vendor_bills                       720
  raw_payroll                          2,860
  raw_journal_entries                 47,880
